In [ ]:

import requests
from bs4 import BeautifulSoup

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

response = requests.get(URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Website:", URL)
print("HTTP status:", response.status_code)
print("Page title:", soup.title.get_text(strip=True) if soup.title else "Not found")
print("Review cards in initial HTML:", len(soup.select("article.review")))

In [ ]:

!pip -q install playwright
!playwright install chromium

In [ ]:

from playwright.sync_api import sync_playwright

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(URL, wait_until="networkidle", timeout=60000)

    page.wait_for_timeout(2000)

    cards = page.locator("article.review")
    count = cards.count()

    print("Page title:", page.title())
    print("Rendered review cards:", count)

    for i in range(min(count, 3)):
        print(f"\n--- Review {i + 1} ---")
        print(cards.nth(i).inner_text())

    browser.close()


In [ ]:
from playwright.async_api import async_playwright
import asyncio

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

async def inspect_reviews():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(2000)

        cards = page.locator("article.review")
        count = await cards.count()

        print("Page title:", await page.title())
        print("Rendered review cards:", count)

        for i in range(min(count, 3)):
            print(f"\n--- Review {i + 1} ---")
            print(await cards.nth(i).inner_text())

        await browser.close()

await inspect_reviews()

In [ ]:
!playwright install-deps chromium

In [ ]:
await inspect_reviews()

In [ ]:
import json
import re
from playwright.async_api import async_playwright

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

async def extract_reviews():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(2000)

        cards = page.locator("article.review")
        reviews = []

        for i in range(await cards.count()):
            card = cards.nth(i)

            def clean(value):
                return value.strip() if value else ""

            reviewer = clean(await card.locator(".name").inner_text())
            date_id = clean(await card.locator(".date").inner_text())
            stars = clean(await card.locator(".stars").inner_text())
            title = clean(await card.locator(".review-title").inner_text())
            review_text = clean(await card.locator(".review-text").inner_text())

            tags = [
                clean(x)
                for x in await card.locator(".review-meta .tag").all_inner_texts()
            ]

            match = re.search(r"#(\d+)", date_id)
            review_id = int(match.group(1)) if match else None

            rating_match = re.search(r"[★☆]", stars)
            rating = stars.count("★") if rating_match else None

            date = date_id.split("·")[0].strip()

            reviews.append({
                "review_id": review_id,
                "reviewer_name": reviewer,
                "date": date,
                "rating": rating,
                "review_title": title,
                "review_text": review_text,
                "sentiment": tags[0] if len(tags) > 0 else "",
                "product_category": tags[1] if len(tags) > 1 else "",
                "order_type": tags[2] if len(tags) > 2 else "",
                "menu_item": tags[3] if len(tags) > 3 else ""
            })

        await browser.close()

    with open("reviews_export.json", "w", encoding="utf-8") as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)

    print(f"Extracted {len(reviews)} reviews.")
    print("Saved to reviews_export.json")
    print("\nFirst extracted review:")
    print(json.dumps(reviews[0], ensure_ascii=False, indent=2))

await extract_reviews()

In [ ]:

from playwright.async_api import async_playwright

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

async def check_all_reviews():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(2000)

        previous_count = 0

        for step in range(10):
            count = await page.locator("article.review").count()
            print(f"Scroll {step}: {count} review cards")

            # Look for buttons that may reveal more reviews
            buttons = await page.get_by_role("button").all_inner_texts()
            if buttons:
                print("Buttons found:", buttons)

            # Scroll to the bottom
            await page.evaluate(
                "() => window.scrollTo(0, document.body.scrollHeight)"
            )
            await page.wait_for_timeout(1000)

            new_count = await page.locator("article.review").count()

            # Stop if scrolling no longer reveals additional cards
            if new_count == count and count == previous_count:
                break

            previous_count = count

        print("\nFinal rendered review count:",
              await page.locator("article.review").count())

        await browser.close()

await check_all_reviews()


In [ ]:

from playwright.async_api import async_playwright

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

async def load_all_reviews():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(1500)

        show_all = page.get_by_role(
            "button", name="Show all 20 reviews"
        )

        print("Before clicking:", await page.locator("article.review").count())

        await show_all.click()
        await page.wait_for_timeout(1000)

        cards = page.locator("article.review")
        count = await cards.count()

        print("After clicking:", count, "review cards")

        # Print the IDs to confirm which reviews are available
        for i in range(count):
            date_id = await cards.nth(i).locator(".date").inner_text()
            print(date_id)

        await browser.close()

await load_all_reviews()


In [ ]:

import json
import re
from playwright.async_api import async_playwright

URL = "https://fanciful-tulumba-d0107d.netlify.app/"

async def extract_all_reviews():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(URL, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(1000)

        # Load all 20 reviews
        await page.get_by_role(
            "button", name="Show all 20 reviews"
        ).click()

        await page.wait_for_timeout(500)

        cards = page.locator("article.review")
        reviews = []

        for i in range(await cards.count()):
            card = cards.nth(i)

            reviewer = (await card.locator(".name").inner_text()).strip()
            date_id = (await card.locator(".date").inner_text()).strip()
            stars = (await card.locator(".stars").inner_text()).strip()
            title = (await card.locator(".review-title").inner_text()).strip()
            review_text = (await card.locator(".review-text").inner_text()).strip()

            tags = [
                value.strip()
                for value in await card.locator(
                    ".review-meta .tag"
                ).all_inner_texts()
            ]

            id_match = re.search(r"#(\d+)", date_id)
            review_id = int(id_match.group(1)) if id_match else None

            reviews.append({
                "review_id": review_id,
                "reviewer_name": reviewer,
                "date": date_id.split("·")[0].strip(),
                "rating": stars.count("★"),
                "review_title": title,
                "review_text": review_text,
                "sentiment": tags[0] if len(tags) > 0 else "",
                "product_category": tags[1] if len(tags) > 1 else "",
                "order_type": tags[2] if len(tags) > 2 else "",
                "menu_item": tags[3] if len(tags) > 3 else ""
            })

        await browser.close()

    with open("reviews_export.json", "w", encoding="utf-8") as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)

    print("Total reviews extracted:", len(reviews))
    print("Unique review IDs:", len({r["review_id"] for r in reviews}))
    print("Saved: reviews_export.json")
    print("\nFirst review:", reviews[0]["review_id"])
    print("Last review:", reviews[-1]["review_id"])

await extract_all_reviews()


In [ ]:

import json
import pandas as pd

with open("reviews_export.json", "r", encoding="utf-8") as f:
    reviews = json.load(f)

df = pd.DataFrame(reviews)

print("Total reviews:", len(df))
print("Unique IDs:", df["review_id"].nunique())
print("\nMissing values:")
print(df.isnull().sum())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

display(df.head())

In [ ]:

%pip -q install -U google-genai


In [ ]:

import google.genai
import google.auth

print("Gemini SDK imported successfully")
print("Google Auth version:", google.auth.__version__)

In [ ]:
from getpass import getpass

GEMINI_API_KEY = getpass("Paste your existing Gemini API key: ")
print("API key received:", bool(GEMINI_API_KEY))

In [ ]:

from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Gemini connection successful."
)

print(response.text)


In [ ]:

import json
from google import genai

# Load the extracted reviews
with open("reviews_export.json", "r", encoding="utf-8") as f:
    reviews = json.load(f)

# Select the first review
review = reviews[0]

prompt = f"""
Analyze this coffee shop customer review.

Review title: {review['review_title']}
Review text: {review['review_text']}
Star rating: {review['rating']}

Return:
1. Overall sentiment: Positive, Neutral, or Negative
2. Food and beverage quality: Positive, Neutral, Negative, or Not mentioned
3. Service quality: Positive, Neutral, Negative, or Not mentioned
4. Ambience: Positive, Neutral, Negative, or Not mentioned
5. A brief explanation for each classification

Return only valid JSON in this format:
{{
  "overall_sentiment": "",
  "food_quality": "",
  "service_quality": "",
  "ambience": "",
  "reasoning": ""
}}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("Review being analyzed:", review["review_title"])
print("\nGemini analysis:\n")
print(response.text)


In [ ]:

import json
import re

# Extract JSON, even if Gemini wraps it in Markdown fences
raw_response = response.text.strip()
raw_response = re.sub(r"^```json\s*", "", raw_response)
raw_response = re.sub(r"\s*```$", "", raw_response)

analysis = json.loads(raw_response)

print("Parsed successfully!")
print(json.dumps(analysis, indent=2, ensure_ascii=False))

# Convert the analysis into a one-row table
import pandas as pd

analysis_df = pd.DataFrame([{
    "review_id": review["review_id"],
    "review_title": review["review_title"],
    "rating": review["rating"],
    **analysis
}])

display(analysis_df)


In [ ]:

import json
from google.genai import types

def analyze_review(review):
    prompt = f"""
You are a customer review analyst for a coffee shop.

Analyze the review below. Use the review text as your primary evidence.
Use the star rating as context, not as the sole basis for sentiment.

Review title: {review['review_title']}
Review text: {review['review_text']}
Star rating: {review['rating']}

Classify:
- overall_sentiment: Positive, Neutral, or Negative
- food_quality: Positive, Neutral, Negative, or Not mentioned
- service_quality: Positive, Neutral, Negative, or Not mentioned
- ambience: Positive, Neutral, Negative, or Not mentioned

For reasoning, briefly explain the classifications using evidence from the review.

Return a JSON object with exactly these keys:
overall_sentiment, food_quality, service_quality, ambience, reasoning
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )

    result = json.loads(response.text)

    return {
        "review_id": review["review_id"],
        "overall_sentiment": result["overall_sentiment"],
        "food_quality": result["food_quality"],
        "service_quality": result["service_quality"],
        "ambience": result["ambience"],
        "reasoning": result["reasoning"]
    }

# Test the function on the first review
test_result = analyze_review(reviews[0])
print(json.dumps(test_result, indent=2, ensure_ascii=False))


In [ ]:

import time
import pandas as pd

analysis_results = []
failed_reviews = []

for i, review in enumerate(reviews, start=1):
    try:
        result = analyze_review(review)
        analysis_results.append(result)
        print(f"{i}/{len(reviews)} completed: Review #{review['review_id']}")

    except Exception as e:
        failed_reviews.append({
            "review_id": review["review_id"],
            "error": str(e)
        })
        print(f"{i}/{len(reviews)} FAILED: Review #{review['review_id']}")

    time.sleep(1)

analysis_df = pd.DataFrame(analysis_results)

print("\nAnalysis complete.")
print("Successful:", len(analysis_results))
print("Failed:", len(failed_reviews))

display(analysis_df.head())


In [ ]:
# Inspect errors from failed reviews
for item in failed_reviews:
    print(f"\nReview #{item['review_id']}")
    print(item["error"][:1000])

# Check the format of successful results
print("\n--- Checking successful results ---")

for result in analysis_results:
    print(
        "Review:", result["review_id"],
        "| reasoning type:", type(result["reasoning"]).__name__
    )

In [ ]:

import pandas as pd

# Make a clean copy of successful results
clean_results = []

for result in analysis_results:
    item = result.copy()

    # Convert nested reasoning dictionaries into readable text
    if isinstance(item.get("reasoning"), dict):
        item["reasoning"] = "; ".join(
            f"{key}: {value}"
            for key, value in item["reasoning"].items()
        )

    clean_results.append(item)

analysis_df = pd.DataFrame(clean_results)

print("Cleaned successful results:", len(analysis_df))
print("Reasoning column types:")
print(analysis_df["reasoning"].apply(type).value_counts())

display(analysis_df[[
    "review_id",
    "overall_sentiment",
    "food_quality",
    "service_quality",
    "ambience",
    "reasoning"
]])



In [ ]:

import time
import json
import pandas as pd

# Keep successful analyses, indexed by review ID
results_by_id = {
    item["review_id"]: item.copy()
    for item in clean_results
}

# Retry only reviews that are still missing
pending_reviews = [
    review for review in reviews
    if review["review_id"] not in results_by_id
]

print("Already completed:", len(results_by_id))
print("Reviews to retry:", len(pending_reviews))

retry_failures = []

for i, review in enumerate(pending_reviews, start=1):
    # Wait before each request to stay below the reported limit
    time.sleep(15)

    try:
        result = analyze_review(review)

        # Normalize nested reasoning values
        if isinstance(result.get("reasoning"), dict):
            result["reasoning"] = "; ".join(
                f"{key}: {value}"
                for key, value in result["reasoning"].items()
            )

        results_by_id[review["review_id"]] = result

        print(
            f"{i}/{len(pending_reviews)} completed: "
            f"Review #{review['review_id']}"
        )

    except Exception as e:
        retry_failures.append({
            "review_id": review["review_id"],
            "error": str(e)
        })
        print(f"{i}/{len(pending_reviews)} FAILED: Review #{review['review_id']}")
        print(str(e)[:250])

# Keep the results in the original review order
analysis_df = pd.DataFrame([
    results_by_id[review["review_id"]]
    for review in reviews
    if review["review_id"] in results_by_id
])

print("\nTotal successful:", len(analysis_df))
print("Still missing:", len(reviews) - len(analysis_df))

display(analysis_df.head())


In [ ]:

import time
import json

for review in reviews:
    review_id = review["review_id"]

    # Skip reviews already analyzed
    if review_id in results_by_id:
        continue

    print(f"Waiting 35 seconds before retrying review #{review_id}...")
    time.sleep(35)

    try:
        result = analyze_review(review)

        if isinstance(result.get("reasoning"), dict):
            result["reasoning"] = "; ".join(
                f"{key}: {value}"
                for key, value in result["reasoning"].items()
            )

        results_by_id[review_id] = result

        # Save progress after each successful review
        with open("analysis_progress.json", "w", encoding="utf-8") as f:
            json.dump(
                list(results_by_id.values()),
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"Review #{review_id}: SUCCESS")

    except Exception as e:
        print(f"Review #{review_id}: FAILED")
        print(str(e)[:500])

analysis_df = pd.DataFrame([
    results_by_id[r["review_id"]]
    for r in reviews
    if r["review_id"] in results_by_id
])

print("\nSuccessful reviews:", len(analysis_df))
print("Still missing:", len(reviews) - len(analysis_df))


In [ ]:
from getpass import getpass

GEMINI_API_KEY_2 = getpass("Enter your second Gemini API key: ")

client2 = genai.Client(api_key=GEMINI_API_KEY_2)

print("Second API key configured:", bool(GEMINI_API_KEY_2))

In [ ]:
response2 = client2.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Second key works."
)

print(response2.text)

In [ ]:

import pandas as pd
import json

# Preserve all successful Gemini analyses
final_results = {
    review_id: result.copy()
    for review_id, result in results_by_id.items()
}

# Fill only the two missing reviews using website-provided sentiment
for review in reviews:
    review_id = review["review_id"]

    if review_id not in final_results:
        final_results[review_id] = {
            "review_id": review_id,
            "overall_sentiment": review["sentiment"],
            "food_quality": "Pending AI analysis",
            "service_quality": "Pending AI analysis",
            "ambience": "Pending AI analysis",
            "reasoning": (
                "Overall sentiment copied from the website. "
                "Aspect analysis pending because the API quota was exhausted."
            )
        }

# Restore the original review order
analysis_df = pd.DataFrame([
    final_results[review["review_id"]]
    for review in reviews
])

print("Total reviews:", len(analysis_df))
print("Gemini-analyzed reviews:", len(results_by_id))
print("Pending aspect analyses:", 20 - len(results_by_id))

display(analysis_df)


In [ ]:

import pandas as pd

# Original extracted review data
original_df = pd.DataFrame(reviews)

# Merge original details with AI analysis
final_df = original_df.merge(
    analysis_df,
    on="review_id",
    how="left",
    suffixes=("_website", "_ai")
)

# Use the AI sentiment where available.
# For the two pending reviews, retain the website sentiment.
final_df["Overall"] = final_df["overall_sentiment"].fillna(
    final_df["sentiment"]
)

# Rename columns for the dashboard
final_df = final_df.rename(columns={
    "reviewer_name": "reviewer_name",
    "review_text": "review",
    "product_category": "product_category",
    "food_quality": "Food Quality",
    "service_quality": "Service",
    "ambience": "Ambience",
    "overall_sentiment": "AI Overall Sentiment",
    "reasoning": "AI Reasoning"
})

print("Final rows:", len(final_df))
print("Unique review IDs:", final_df["review_id"].nunique())
print("\nColumns available for Excel:")
print(final_df.columns.tolist())

display(final_df.head())


In [ ]:

# Export the merged dataset to Excel
output_file = "kaapi_junction_analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="Review Analysis")

    # Add a separate summary sheet for quick review
    summary_df = pd.DataFrame({
        "Metric": [
            "Total reviews",
            "Gemini-analyzed reviews",
            "Pending aspect analyses",
            "Unique review IDs"
        ],
        "Value": [
            len(final_df),
            len(results_by_id),
            len(reviews) - len(results_by_id),
            final_df["review_id"].nunique()
        ]
    })
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

print("Excel file created:", output_file)
print("Total reviews exported:", len(final_df))


In [ ]:

from google.colab import files

files.download("kaapi_junction_analysis.xlsx")


In [ ]:

print("Dashboard data validation")
print("--------------------------")
print("Total rows:", len(final_df))
print("Unique review IDs:", final_df["review_id"].nunique())
print("Missing review text:", final_df["review"].isna().sum())
print("Missing ratings:", final_df["rating"].isna().sum())

print("\nSentiment counts used by the dashboard:")
print(final_df["Overall"].value_counts())

print("\nAverage rating:", round(final_df["rating"].mean(), 2))



In [ ]:

import json
from google.genai import types

def generate_manager_response(review):
    prompt = f"""
You manage Kaapi Junction, a coffee shop.

Write a concise, natural response to this customer review.

Review title: {review['review_title']}
Review text: {review['review_text']}
Star rating: {review['rating']}/5
Website sentiment: {review['sentiment']}
Menu item: {review['menu_item']}
Order type: {review['order_type']}

Instructions:
- Address the customer's actual experience.
- For positive reviews, express genuine appreciation.
- For neutral reviews, acknowledge their feedback constructively.
- For negative reviews, show empathy and acknowledge the specific issue.
- Do not argue, blame the customer, invent facts, or promise actions
  that have not been confirmed.
- Keep the response between 30 and 60 words.
- Return only a JSON object with one key: manager_response.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.4
        )
    )

    result = json.loads(response.text)
    return result["manager_response"]

print("Manager-response function defined.")
print("No API calls made.")


In [ ]:

%pip -q install groq


In [ ]:

from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Paste your Groq API key: ")
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client configured:", True)


In [ ]:

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection successful."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)



In [ ]:

models = groq_client.models.list()

for model in models.data:
    if getattr(model, "active", True):
        print(model.id)


In [ ]:

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection successful."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)


In [ ]:

import json
import time

def analyze_review_groq(review):
    prompt = f"""
Analyze this coffee shop customer review.

Title: {review['review_title']}
Review: {review['review_text']}
Rating: {review['rating']}/5

Classify:
- overall_sentiment: Positive, Neutral, or Negative
- food_quality: Positive, Neutral, Negative, or Not mentioned
- service_quality: Positive, Neutral, Negative, or Not mentioned
- ambience: Positive, Neutral, Negative, or Not mentioned

Use the review text as evidence. Return only a JSON object with these keys:
overall_sentiment, food_quality, service_quality, ambience, reasoning
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a customer review analyst. Return valid JSON only."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)

    return {
        "review_id": review["review_id"],
        "overall_sentiment": result["overall_sentiment"],
        "food_quality": result["food_quality"],
        "service_quality": result["service_quality"],
        "ambience": result["ambience"],
        "reasoning": result["reasoning"]
    }


# Analyze only reviews not already present in results_by_id
for review in reviews:
    review_id = review["review_id"]

    if review_id in results_by_id:
        continue

    try:
        result = analyze_review_groq(review)
        results_by_id[review_id] = result

        with open("analysis_progress.json", "w", encoding="utf-8") as f:
            json.dump(
                list(results_by_id.values()),
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"Review #{review_id}: completed with Groq")

    except Exception as e:
        print(f"Review #{review_id}: FAILED - {str(e)[:300]}")

    time.sleep(2)

print("\nTotal reviews analyzed:", len(results_by_id))
print("Remaining:", len(reviews) - len(results_by_id))


In [ ]:

import pandas as pd
import json

# Rebuild analysis table in original review order
analysis_df = pd.DataFrame([
    results_by_id[review["review_id"]]
    for review in reviews
])

# Normalize any nested reasoning values
for i, value in analysis_df["reasoning"].items():
    if isinstance(value, dict):
        analysis_df.at[i, "reasoning"] = "; ".join(
            f"{key}: {text}"
            for key, text in value.items()
        )

print("Total analyzed:", len(analysis_df))
print("Unique review IDs:", analysis_df["review_id"].nunique())
print("\nOverall sentiment:")
print(analysis_df["overall_sentiment"].value_counts())

display(analysis_df.head())


In [ ]:

import json
import os
import time

def generate_manager_response_groq(review):
    prompt = f"""
Write a concise, professional response from Kaapi Junction to this customer.

Review title: {review['review_title']}
Review text: {review['review_text']}
Rating: {review['rating']}/5
Sentiment: {review['sentiment']}
Menu item: {review['menu_item']}
Order type: {review['order_type']}

Instructions:
- Positive review: thank the customer specifically.
- Neutral review: acknowledge the feedback constructively.
- Negative review: show empathy and address the stated issue.
- Do not argue, invent facts, or promise an unconfirmed action.
- Keep the response between 30 and 60 words.
- Return JSON with one key: manager_response.
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You write thoughtful customer-service responses. Return valid JSON only."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.4,
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)
    return result["manager_response"]


PROGRESS_FILE = "manager_responses_progress.json"

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        saved_responses = json.load(f)
else:
    saved_responses = {}

for i, review in enumerate(reviews, start=1):
    review_id = str(review["review_id"])

    if review_id in saved_responses:
        print(f"Skipping review #{review_id}: already completed")
        continue

    try:
        manager_response = generate_manager_response_groq(review)
        saved_responses[review_id] = manager_response

        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(saved_responses, f, ensure_ascii=False, indent=2)

        print(f"{i}/{len(reviews)} completed: Review #{review_id}")
        time.sleep(2)

    except Exception as e:
        print(f"Review #{review_id} failed: {str(e)[:300]}")
        print("Progress saved. Stop and share the error before retrying.")
        break

print("\nManager responses saved:", len(saved_responses))
print("Remaining:", len(reviews) - len(saved_responses))


In [ ]:

import pandas as pd
import json

# Load the saved manager responses
with open("manager_responses_progress.json", "r", encoding="utf-8") as f:
    saved_responses = json.load(f)

# Rebuild analysis data in original review order
analysis_df = pd.DataFrame([
    results_by_id[review["review_id"]]
    for review in reviews
])

# Normalize reasoning for Excel
for i, value in analysis_df["reasoning"].items():
    if isinstance(value, dict):
        analysis_df.at[i, "reasoning"] = "; ".join(
            f"{key}: {text}" for key, text in value.items()
        )

# Original review details
original_df = pd.DataFrame(reviews)

# Merge original details and AI analysis
final_df = original_df.merge(
    analysis_df,
    on="review_id",
    how="left",
    suffixes=("_website", "_ai")
)

# Add manager responses
final_df["manager_response"] = final_df["review_id"].map(
    lambda rid: saved_responses.get(str(rid), "")
)

# Use AI sentiment, retaining the website label as a fallback
final_df["Overall"] = final_df["overall_sentiment"].fillna(
    final_df["sentiment"]
)

# Create a dedicated response-review sheet
response_df = final_df[[
    "review_id",
    "reviewer_name",
    "rating",
    "review_title",
    "review",
    "sentiment",
    "manager_response"
]].copy()

# Export workbook
output_file = "kaapi_junction_final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="Review Analysis")
    response_df.to_excel(writer, index=False, sheet_name="Manager Responses")

    summary_df = pd.DataFrame({
        "Metric": [
            "Total reviews",
            "Unique review IDs",
            "Completed AI analyses",
            "Manager responses generated",
            "Average rating"
        ],
        "Value": [
            len(final_df),
            final_df["review_id"].nunique(),
            len(results_by_id),
            len(saved_responses),
            round(final_df["rating"].mean(), 2)
        ]
    })
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

print("Workbook created:", output_file)
print("Reviews:", len(final_df))
print("Manager responses:", final_df["manager_response"].notna().sum())
print("Unique IDs:", final_df["review_id"].nunique())


In [ ]:
print(final_df.columns.tolist())


In [ ]:

import pandas as pd
import json

# Load the saved manager responses
with open("manager_responses_progress.json", "r", encoding="utf-8") as f:
    saved_responses = json.load(f)

# Rebuild the final dataset
analysis_df = pd.DataFrame([
    results_by_id[review["review_id"]]
    for review in reviews
])

# Convert nested reasoning objects into readable text
for i, value in analysis_df["reasoning"].items():
    if isinstance(value, dict):
        analysis_df.at[i, "reasoning"] = "; ".join(
            f"{key}: {text}" for key, text in value.items()
        )

original_df = pd.DataFrame(reviews)

final_df = original_df.merge(
    analysis_df,
    on="review_id",
    how="left",
    suffixes=("_website", "_ai")
)

# Add generated manager responses
final_df["manager_response"] = final_df["review_id"].map(
    lambda rid: saved_responses.get(str(rid), "")
)

# Standardize the review text column for export
final_df["review"] = final_df["review_text"]

# Create the manager response sheet
response_df = final_df[
    [
        "review_id",
        "reviewer_name",
        "rating",
        "review_title",
        "review",
        "sentiment",
        "manager_response"
    ]
].copy()

# Create a summary sheet
summary_df = pd.DataFrame({
    "Metric": [
        "Total reviews",
        "Average rating",
        "Positive reviews",
        "Neutral reviews",
        "Negative reviews",
        "Manager responses generated"
    ],
    "Value": [
        len(final_df),
        round(final_df["rating"].mean(), 2),
        int((final_df["Overall"].str.lower() == "positive").sum()),
        int((final_df["Overall"].str.lower() == "neutral").sum()),
        int((final_df["Overall"].str.lower() == "negative").sum()),
        int(final_df["manager_response"].fillna("").str.strip().ne("").sum())
    ]
})

# Export all sheets to Excel
output_file = "kaapi_junction_final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="Review Analysis", index=False)
    response_df.to_excel(writer, sheet_name="Manager Responses", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Excel workbook created:", output_file)
print("Reviews:", len(final_df))
print("Manager responses:", response_df["manager_response"].fillna("").str.strip().ne("").sum())

# Download the workbook
from google.colab import files
files.download(output_file)


In [ ]:

import pandas as pd
import json

# Load manager responses
with open("manager_responses_progress.json", "r", encoding="utf-8") as f:
    saved_responses = json.load(f)

# Rebuild analysis data
analysis_df = pd.DataFrame([
    results_by_id[review["review_id"]]
    for review in reviews
])

# Convert nested reasoning dictionaries to readable text
if "reasoning" in analysis_df.columns:
    for i, value in analysis_df["reasoning"].items():
        if isinstance(value, dict):
            analysis_df.at[i, "reasoning"] = "; ".join(
                f"{key}: {text}" for key, text in value.items()
            )

# Merge original reviews with AI analysis
original_df = pd.DataFrame(reviews)

final_df = original_df.merge(
    analysis_df,
    on="review_id",
    how="left",
    suffixes=("_website", "_ai")
)

# Add manager responses
final_df["manager_response"] = final_df["review_id"].map(
    lambda rid: saved_responses.get(str(rid), "")
)

# Ensure review text exists
if "review_text" in final_df.columns:
    final_df["review"] = final_df["review_text"]
elif "review_text_website" in final_df.columns:
    final_df["review"] = final_df["review_text_website"]
else:
    raise KeyError("Review text column not found.")

# Determine the sentiment column safely
if "overall_sentiment" in final_df.columns:
    final_df["Overall"] = final_df["overall_sentiment"]
elif "sentiment_ai" in final_df.columns:
    final_df["Overall"] = final_df["sentiment_ai"]
elif "sentiment" in final_df.columns:
    final_df["Overall"] = final_df["sentiment"]
elif "sentiment_website" in final_df.columns:
    final_df["Overall"] = final_df["sentiment_website"]
else:
    raise KeyError("No sentiment column found.")

# Create manager response sheet
response_df = final_df[
    [
        "review_id",
        "reviewer_name",
        "rating",
        "review_title",
        "review",
        "Overall",
        "manager_response"
    ]
].rename(columns={"Overall": "sentiment"})

# Create summary sheet
sentiments = final_df["Overall"].fillna("").str.strip().str.lower()

summary_df = pd.DataFrame({
    "Metric": [
        "Total reviews",
        "Average rating",
        "Positive reviews",
        "Neutral reviews",
        "Negative reviews",
        "Manager responses generated"
    ],
    "Value": [
        len(final_df),
        round(final_df["rating"].mean(), 2),
        int((sentiments == "positive").sum()),
        int((sentiments == "neutral").sum()),
        int((sentiments == "negative").sum()),
        int(final_df["manager_response"].fillna("").str.strip().ne("").sum())
    ]
})

# Export workbook
output_file = "kaapi_junction_final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="Review Analysis", index=False)
    response_df.to_excel(writer, sheet_name="Manager Responses", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Workbook created successfully.")
print("Reviews:", len(final_df))
print("Responses:", int(final_df["manager_response"].fillna("").str.strip().ne("").sum()))
print("\nSummary:")
print(summary_df.to_string(index=False))

from google.colab import files
files.download(output_file)


In [ ]:

import json

# Build the JSON payload expected by the manager-response importer
manager_responses_json = [
    {
        "review_id": int(review["review_id"]),
        "manager_response": saved_responses.get(str(review["review_id"]), "")
    }
    for review in reviews
]

# Validate the output
assert len(manager_responses_json) == 20
assert all(item["manager_response"].strip() for item in manager_responses_json)

output_file = "kaapi_junction_manager_responses.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(manager_responses_json, f, ensure_ascii=False, indent=2)

print(f"Created {output_file}")
print(f"Responses: {len(manager_responses_json)}")

from google.colab import files
files.download(output_file)



In [ ]:

import json

enriched_reviews = []

for review in reviews:
    rid = review["review_id"]
    analysis = results_by_id[rid]

    record = {
        # Website's expected dashboard fields
        "review_id": rid,
        "reviewer_name": review["reviewer_name"],
        "review_date": review["date"],
        "product_category": review["product_category"],
        "rating": review["rating"],
        "review_title": review["review_title"],
        "review_text": review["review_text"],
        "sentiment": review["sentiment"],
        "task2_sentiment": review["sentiment"],
        "dish_ordered": review["menu_item"],
        "visit_type": review["order_type"],

        # Preserve the website's original aspect scores
        "food_quality": review.get("food_quality"),
        "service": review.get("service"),
        "ambience": review.get("ambience"),
        "value_for_money": review.get("value_for_money"),
        "would_recommend": review.get("would_recommend"),

        # Add our AI analysis separately
        "ai_overall_sentiment": analysis.get("overall_sentiment"),
        "ai_food_quality": analysis.get("food_quality"),
        "ai_service_quality": analysis.get("service_quality"),
        "ai_ambience": analysis.get("ambience"),
        "ai_reasoning": analysis.get("reasoning"),

        # Add the generated manager response
        "manager_response": saved_responses.get(str(rid), "")
    }

    enriched_reviews.append(record)

output_file = "kaapi_junction_enriched_dashboard.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(enriched_reviews, f, ensure_ascii=False, indent=2)

assert len(enriched_reviews) == 20
assert all(r["manager_response"] for r in enriched_reviews)

print("Enriched JSON created:", output_file)
print("Review records:", len(enriched_reviews))
print("Manager responses:", sum(bool(r["manager_response"]) for r in enriched_reviews))

from google.colab import files
files.download(output_file)


In [ ]:
from pathlib import Path

project_root = Path("/content/customer-review-analytics")

folders = [
    "notebooks",
    "data",
    "outputs"
]

for folder in folders:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created:")
for path in sorted(project_root.rglob("*")):
    if path.is_dir():
        print(path.relative_to(project_root))

In [ ]:

from pathlib import Path

project_root = Path("/content/customer-review-analytics")

# 1. README
readme = """# Customer Review Analytics

An AI-powered customer review analytics project built using Python and Google Colab.

## Features
- Extracts reviews from a demonstration restaurant website
- Classifies overall sentiment using LLM APIs
- Analyzes food quality, service, and ambience
- Generates customer-service response suggestions
- Exports review analysis to Excel
- Supports importing manager responses into the demonstration website
- Visualizes ratings, sentiment, and review categories in a dashboard

## Technologies
- Python
- Playwright
- Pandas
- Google Gemini API
- Groq API
- Excel (openpyxl)
- Google Colab

## Project Structure
- `notebooks/` - analysis notebook
- `data/` - sample review data
- `outputs/` - example analysis exports

## Setup
Open the notebook in Google Colab and provide your own API keys through
Colab Secrets or secure runtime input. Never commit API keys.

## Data Notice
The restaurant reviews are demonstration-project data and should not be
treated as verified real customer feedback.

## Limitations
API quotas and model availability can affect processing. The notebook
may use different LLM providers for different reviews.

## License
MIT
"""

(project_root / "README.md").write_text(readme, encoding="utf-8")

# 2. Python dependencies
requirements = """pandas
playwright
google-genai
groq
openpyxl
"""

(project_root / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

# 3. Git ignore rules
gitignore = """# Secrets and environment files
.env
.env.*
!.env.example
*api_key*
*secret*
secrets.json
credentials.json
token.json

# Python and notebook runtime files
__pycache__/
*.py[cod]
.ipynb_checkpoints/

# Temporary and raw exports
*_progress.json
reviews_export.json
kaapi_junction_reviews_export.json
*.log
.DS_Store
"""

(project_root / ".gitignore").write_text(
    gitignore, encoding="utf-8"
)

print("Created:")
for name in ["README.md", "requirements.txt", ".gitignore"]:
    print("-", name)


In [ ]:
from pathlib import Path

project_root = Path("/content/customer-review-analytics")

for name in ["README.md", "requirements.txt", ".gitignore"]:
    path = project_root / name
    print(f"\n--- {name} ---")
    print(path.read_text(encoding="utf-8")[:500])